# Large-Scale Systemic-Risk Simulation (Interbank)

This notebook implements a Monte Carlo contagion workflow to label banks as systemically important (`1`) or not (`0`) for each quarter in the dataset.

## Pipeline
- Load quarter node/edge data (`datasets/nodes`, `datasets/edges`)
- Simulate contagion with two-stage defaults (liquidity first, then exposure)
- Run mixed shock scenarios (single + multi bank seeds)
- Compute trigger impact score per node
- Label top quantile nodes as systemically important
- Export labels, scenario logs, and summary tables


In [ ]:
# Optional: install missing dependencies in your notebook environment
# Uncomment and run if needed.
# %pip install -U numpy pandas pyarrow matplotlib seaborn networkx


In [ ]:
from __future__ import annotations

import copy
import hashlib
import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
except Exception:
    HAS_PLOTS = False


In [ ]:
def resolve_project_root(start: Path | None = None) -> Path:
    """Find project root by searching for datasets folder."""
    current = Path.cwd() if start is None else Path(start)
    for candidate in [current, *current.parents]:
        if (candidate / "datasets" / "nodes").exists() and (candidate / "datasets" / "edges").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing datasets/nodes and datasets/edges")


PROJECT_ROOT = resolve_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
for subdir in ["systemic_labels", "scenario_logs", "summaries", "plots"]:
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)


CONFIG = {
    "quarters": [f"{year}Q{quarter}" for year in range(2016, 2023) for quarter in range(1, 5)] + ["2023Q1"],
    "n_scenarios": 10_000,
    "single_share": 0.5,
    "multi_share": 0.5,
    "multi_seed_fraction": 0.05,
    "shock_range": (0.20, 0.60),
    "capital_ratio": 0.08,
    "liquidity_ratio": 0.30,
    "label_quantile": 0.90,
    "lgd": 1.0,
    "rng_seed": 20260303,
    "eps": 1e-9,
    "max_rounds": None,
    "negative_weight_policy": "split_by_sign",
}


VALID_NEGATIVE_WEIGHT_POLICIES = {"split_by_sign", "ignore_negative"}
LOADED_QUARTER_LOGS: set[tuple[str, str]] = set()


assert abs(CONFIG["single_share"] + CONFIG["multi_share"] - 1.0) < 1e-12
assert len(CONFIG["quarters"]) == 29
assert CONFIG["negative_weight_policy"] in VALID_NEGATIVE_WEIGHT_POLICIES
print(f"Project root: {PROJECT_ROOT}")
print(f"Configured quarters: {len(CONFIG['quarters'])}")
print(f"Negative weight policy: {CONFIG['negative_weight_policy']}")


In [ ]:
def quarter_seed(base_seed: int, quarter: str) -> int:
    """Derive a deterministic per-quarter seed."""
    digest = hashlib.md5(quarter.encode("utf-8")).hexdigest()
    return base_seed + int(digest[:8], 16)


def apply_negative_weight_policy(edges_df: pd.DataFrame, policy: str) -> pd.DataFrame:
    """Normalize signed edge weights according to the configured policy."""
    if policy not in VALID_NEGATIVE_WEIGHT_POLICIES:
        raise ValueError(
            f"Unknown negative_weight_policy='{policy}'. Expected one of {sorted(VALID_NEGATIVE_WEIGHT_POLICIES)}"
        )

    edges_df = edges_df.copy()
    negative_mask = edges_df["Weights"] < 0

    if policy == "ignore_negative":
        return edges_df.loc[~negative_mask].reset_index(drop=True)

    non_negative = edges_df.loc[~negative_mask].copy()
    negative = edges_df.loc[negative_mask].copy()
    if not negative.empty:
        swapped_source = negative["Targetid"].to_numpy()
        swapped_target = negative["Sourceid"].to_numpy()
        negative["Sourceid"] = swapped_source
        negative["Targetid"] = swapped_target
        negative["Weights"] = negative["Weights"].abs()

    normalized = pd.concat([non_negative, negative], ignore_index=True)
    return normalized.reset_index(drop=True)


def load_quarter_data(quarter: str, cfg: dict[str, Any] | None = None) -> tuple[pd.DataFrame, pd.DataFrame, dict[int, int]]:
    """Load one quarter of nodes and edges and return node index mapping."""
    cfg = CONFIG if cfg is None else cfg
    policy = cfg.get("negative_weight_policy", "split_by_sign")

    node_path = PROJECT_ROOT / "datasets" / "nodes" / f"{quarter}.csv"
    edge_path = PROJECT_ROOT / "datasets" / "edges" / f"edge_{quarter}.csv"

    if not node_path.exists():
        raise FileNotFoundError(f"Missing node file: {node_path}")
    if not edge_path.exists():
        raise FileNotFoundError(f"Missing edge file: {edge_path}")

    nodes_df = pd.read_csv(node_path)
    edges_df = pd.read_csv(edge_path)

    required_edge_cols = {"Sourceid", "Targetid", "Weights"}
    if not required_edge_cols.issubset(set(edges_df.columns)):
        raise ValueError(f"Edge file {edge_path} missing required columns: {required_edge_cols}")
    if "index" not in nodes_df.columns:
        raise ValueError(f"Node file {node_path} missing 'index' column")

    nodes_df = nodes_df.copy()
    edges_df = edges_df.copy()

    nodes_df["index"] = nodes_df["index"].astype(int)
    edges_df["Sourceid"] = edges_df["Sourceid"].astype(int)
    edges_df["Targetid"] = edges_df["Targetid"].astype(int)
    edges_df["Weights"] = edges_df["Weights"].astype(float)

    n_negative_input_edges = int((edges_df["Weights"] < 0).sum())
    edges_df = apply_negative_weight_policy(edges_df, policy)

    if nodes_df["index"].duplicated().any():
        dup = nodes_df[nodes_df["index"].duplicated()]["index"].head().tolist()
        raise ValueError(f"Duplicate node ids in {quarter}: {dup}")

    node_ids = nodes_df["index"].to_numpy()
    node_index_map = {int(bank_id): idx for idx, bank_id in enumerate(node_ids)}

    in_universe = edges_df["Sourceid"].isin(node_index_map) & edges_df["Targetid"].isin(node_index_map)
    filtered_edges_df = edges_df.loc[in_universe].reset_index(drop=True)

    diagnostics = {
        "quarter": quarter,
        "negative_weight_policy": policy,
        "n_negative_input_edges": n_negative_input_edges,
        "n_edges_after_policy": int(len(filtered_edges_df)),
    }
    filtered_edges_df.attrs["negative_weight_diagnostics"] = diagnostics

    log_key = (quarter, policy)
    if log_key not in LOADED_QUARTER_LOGS:
        print(
            f"[{quarter}] policy={policy} raw_negative_edges={n_negative_input_edges} "
            f"edges_after_policy={len(filtered_edges_df)}"
        )
        LOADED_QUARTER_LOGS.add(log_key)

    return nodes_df, filtered_edges_df, node_index_map


def build_network_state(nodes_df: pd.DataFrame, edges_df: pd.DataFrame, node_index_map: dict[int, int], cfg: dict[str, Any]) -> dict[str, Any]:
    """Build simulation-ready arrays for one quarter."""
    node_ids = nodes_df["index"].astype(int).to_numpy()
    n_nodes = len(node_ids)

    src = edges_df["Sourceid"].map(node_index_map).astype(int).to_numpy()
    tgt = edges_df["Targetid"].map(node_index_map).astype(int).to_numpy()
    w = edges_df["Weights"].astype(float).to_numpy()

    if np.any(w < 0):
        policy = cfg.get("negative_weight_policy", "split_by_sign")
        raise ValueError(
            "Negative edge weights remain after preprocessing. "
            f"Check negative_weight_policy='{policy}'."
        )

    out_strength = np.bincount(src, weights=w, minlength=n_nodes)
    in_strength = np.bincount(tgt, weights=w, minlength=n_nodes)

    capital_buffer = np.maximum(cfg["capital_ratio"] * out_strength, cfg["eps"])
    liquidity_buffer = np.maximum(cfg["liquidity_ratio"] * in_strength, cfg["eps"])

    return {
        "node_ids": node_ids,
        "n_nodes": n_nodes,
        "n_edges": int(len(w)),
        "src": src,
        "tgt": tgt,
        "weight": w,
        "out_strength": out_strength,
        "in_strength": in_strength,
        "capital_buffer": capital_buffer,
        "liquidity_buffer": liquidity_buffer,
        "total_system_out_strength": float(np.maximum(out_strength.sum(), cfg["eps"])),
    }


In [ ]:
def _sum_weights_by_target(src: np.ndarray, tgt: np.ndarray, w: np.ndarray, defaulted_sources: np.ndarray, n_nodes: int) -> np.ndarray:
    """For liquidity channel: defaults at source withdraw funding from targets."""
    if defaulted_sources.size == 0:
        return np.zeros(n_nodes, dtype=float)
    mask = np.isin(src, defaulted_sources, assume_unique=False)
    if not mask.any():
        return np.zeros(n_nodes, dtype=float)
    return np.bincount(tgt[mask], weights=w[mask], minlength=n_nodes)


def _sum_weights_by_source(src: np.ndarray, tgt: np.ndarray, w: np.ndarray, defaulted_targets: np.ndarray, n_nodes: int) -> np.ndarray:
    """For exposure channel: defaults at target create losses for source lenders."""
    if defaulted_targets.size == 0:
        return np.zeros(n_nodes, dtype=float)
    mask = np.isin(tgt, defaulted_targets, assume_unique=False)
    if not mask.any():
        return np.zeros(n_nodes, dtype=float)
    return np.bincount(src[mask], weights=w[mask], minlength=n_nodes)


def run_scenario(state_dict: dict[str, Any], scenario_spec: dict[str, Any], rng: np.random.Generator) -> dict[str, Any]:
    """Run one two-stage cascade scenario and return metrics."""
    n = state_dict["n_nodes"]
    src = state_dict["src"]
    tgt = state_dict["tgt"]
    w = state_dict["weight"]

    in_strength = state_dict["in_strength"]
    capital_buffer = state_dict["capital_buffer"]
    liquidity_buffer = state_dict["liquidity_buffer"]

    lgd = float(scenario_spec.get("lgd", 1.0))
    max_rounds = int(scenario_spec.get("max_rounds", n))

    seed_banks = np.array(scenario_spec.get("seed_banks", []), dtype=int)
    shock_values = np.array(scenario_spec.get("shock_values", []), dtype=float)

    if seed_banks.size != shock_values.size:
        raise ValueError("seed_banks and shock_values must have same length")

    alive = np.ones(n, dtype=bool)
    defaulted = np.zeros(n, dtype=bool)

    liquidity_loss = np.zeros(n, dtype=float)
    exposure_loss = np.zeros(n, dtype=float)

    if seed_banks.size > 0:
        np.add.at(liquidity_loss, seed_banks, shock_values * in_strength[seed_banks])

    new_defaults_prev = np.array([], dtype=int)
    unprocessed_credit_defaults = np.array([], dtype=int)
    default_count_history = []

    rounds_executed = 0

    for round_id in range(1, max_rounds + 1):
        rounds_executed = round_id

        liquidity_increment = _sum_weights_by_target(src, tgt, w, new_defaults_prev, n)
        liquidity_loss += liquidity_increment

        stage_a_defaults = alive & (liquidity_loss > liquidity_buffer)
        if stage_a_defaults.any():
            alive[stage_a_defaults] = False
            defaulted[stage_a_defaults] = True

        stage_a_idx = np.flatnonzero(stage_a_defaults)
        if stage_a_idx.size > 0:
            if unprocessed_credit_defaults.size == 0:
                unprocessed_credit_defaults = stage_a_idx
            else:
                unprocessed_credit_defaults = np.unique(np.concatenate([unprocessed_credit_defaults, stage_a_idx]))

        exposure_increment = _sum_weights_by_source(src, tgt, w, unprocessed_credit_defaults, n)
        exposure_loss += lgd * exposure_increment

        stage_b_defaults = alive & (exposure_loss > capital_buffer)
        if stage_b_defaults.any():
            alive[stage_b_defaults] = False
            defaulted[stage_b_defaults] = True

        stage_b_idx = np.flatnonzero(stage_b_defaults)

        if stage_b_idx.size > 0:
            unprocessed_credit_defaults = stage_b_idx
        else:
            unprocessed_credit_defaults = np.array([], dtype=int)

        new_defaults = stage_a_defaults | stage_b_defaults
        new_defaults_prev = np.flatnonzero(new_defaults)

        default_count_history.append(int(defaulted.sum()))

        if new_defaults_prev.size == 0:
            break

    default_count = int(defaulted.sum())
    default_share = default_count / n
    loss_exposure = float(exposure_loss.sum())
    loss_liquidity = float(liquidity_loss.sum())

    impact_score = 0.7 * default_share + 0.3 * (loss_exposure / state_dict["total_system_out_strength"])

    return {
        "scenario_id": scenario_spec.get("scenario_id"),
        "shock_type": scenario_spec.get("shock_type", "unknown"),
        "seed_banks": seed_banks.tolist(),
        "shock_values": shock_values.tolist(),
        "default_count": default_count,
        "default_share": default_share,
        "loss_exposure": loss_exposure,
        "loss_liquidity": loss_liquidity,
        "impact_score": float(impact_score),
        "rounds": rounds_executed,
        "default_count_history": default_count_history,
    }


In [ ]:
def _parse_list_column(value: Any) -> list:
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        return json.loads(value)
    return list(value)


def compute_trigger_scores(scenario_df: pd.DataFrame, n_nodes: int) -> pd.DataFrame:
    """Compute average impact score for each bank when it appears in initial seed set."""
    score_sum = np.zeros(n_nodes, dtype=float)
    score_count = np.zeros(n_nodes, dtype=int)

    for _, row in scenario_df.iterrows():
        seeds = _parse_list_column(row["seed_banks"])
        if not seeds:
            continue
        impact = float(row["impact_score"])
        seed_idx = np.array(seeds, dtype=int)
        np.add.at(score_sum, seed_idx, impact)
        np.add.at(score_count, seed_idx, 1)

    trigger_score = np.divide(
        score_sum,
        np.maximum(score_count, 1),
        out=np.zeros_like(score_sum),
        where=score_count > 0,
    )

    return pd.DataFrame(
        {
            "node_idx": np.arange(n_nodes, dtype=int),
            "seed_scenario_count": score_count,
            "trigger_impact_score": trigger_score,
        }
    )


def _sample_scenario_spec(n_nodes: int, scenario_id: int, shock_type: str, cfg: dict[str, Any], rng: np.random.Generator) -> dict[str, Any]:
    low, high = cfg["shock_range"]

    if shock_type == "single":
        seeds = rng.integers(0, n_nodes, size=1, endpoint=False)
    elif shock_type == "multi":
        k = max(1, math.ceil(cfg["multi_seed_fraction"] * n_nodes))
        seeds = rng.choice(n_nodes, size=k, replace=False)
    else:
        raise ValueError(f"Unknown shock type: {shock_type}")

    shocks = rng.uniform(low, high, size=len(seeds))
    return {
        "scenario_id": scenario_id,
        "shock_type": shock_type,
        "seed_banks": seeds.tolist(),
        "shock_values": shocks.tolist(),
        "lgd": cfg["lgd"],
        "max_rounds": cfg["max_rounds"] or n_nodes,
    }


In [ ]:
def _write_scenario_log(df: pd.DataFrame, output_path: Path) -> Path:
    """Write scenario logs as parquet when available, else CSV fallback."""
    try:
        df.to_parquet(output_path, index=False)
        return output_path
    except Exception as exc:
        fallback_path = output_path.with_suffix(".csv")
        print(f"Parquet write failed ({exc}). Falling back to CSV: {fallback_path.name}")
        df.to_csv(fallback_path, index=False)
        return fallback_path


def run_quarter_simulations(quarter: str, cfg: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Execute all simulations for a quarter and return labels, scenarios, and summary."""
    nodes_df, edges_df, node_index_map = load_quarter_data(quarter, cfg)
    state = build_network_state(nodes_df, edges_df, node_index_map, cfg)

    diagnostics = edges_df.attrs.get("negative_weight_diagnostics", {})
    n_negative_input_edges = int(diagnostics.get("n_negative_input_edges", 0))
    n_edges_after_policy = int(diagnostics.get("n_edges_after_policy", len(edges_df)))
    negative_weight_policy = str(diagnostics.get("negative_weight_policy", cfg.get("negative_weight_policy", "split_by_sign")))

    n_nodes = state["n_nodes"]
    n_scenarios = int(cfg["n_scenarios"])

    n_single = int(round(n_scenarios * cfg["single_share"]))
    n_multi = n_scenarios - n_single
    scenario_types = ["single"] * n_single + ["multi"] * n_multi

    rng = np.random.default_rng(quarter_seed(cfg["rng_seed"], quarter))
    rng.shuffle(scenario_types)

    records: list[dict[str, Any]] = []

    for scenario_id, shock_type in enumerate(scenario_types):
        spec = _sample_scenario_spec(n_nodes, scenario_id, shock_type, cfg, rng)
        result = run_scenario(state, spec, rng)
        records.append(
            {
                "scenario_id": result["scenario_id"],
                "shock_type": result["shock_type"],
                "seed_banks": json.dumps(result["seed_banks"]),
                "shock_values": json.dumps(result["shock_values"]),
                "default_count": result["default_count"],
                "default_share": result["default_share"],
                "loss_exposure": result["loss_exposure"],
                "loss_liquidity": result["loss_liquidity"],
                "impact_score": result["impact_score"],
                "rounds": result["rounds"],
            }
        )

    scenario_df = pd.DataFrame(records)

    trigger_df = compute_trigger_scores(scenario_df, n_nodes)
    missing_nodes = trigger_df.loc[trigger_df["seed_scenario_count"] == 0, "node_idx"].to_numpy(dtype=int)

    backfill_records = []
    if missing_nodes.size > 0:
        fixed_shock = float(np.mean(cfg["shock_range"]))
        for node_idx in missing_nodes:
            backfill_spec = {
                "scenario_id": f"backfill_{node_idx}",
                "shock_type": "backfill_single",
                "seed_banks": [int(node_idx)],
                "shock_values": [fixed_shock],
                "lgd": cfg["lgd"],
                "max_rounds": cfg["max_rounds"] or n_nodes,
            }
            result = run_scenario(state, backfill_spec, rng)
            backfill_records.append(
                {
                    "scenario_id": result["scenario_id"],
                    "shock_type": result["shock_type"],
                    "seed_banks": json.dumps(result["seed_banks"]),
                    "shock_values": json.dumps(result["shock_values"]),
                    "default_count": result["default_count"],
                    "default_share": result["default_share"],
                    "loss_exposure": result["loss_exposure"],
                    "loss_liquidity": result["loss_liquidity"],
                    "impact_score": result["impact_score"],
                    "rounds": result["rounds"],
                }
            )

    if backfill_records:
        scenario_df = pd.concat([scenario_df, pd.DataFrame(backfill_records)], ignore_index=True)
        trigger_df = compute_trigger_scores(scenario_df, n_nodes)

    raw_quantile_threshold = float(np.quantile(trigger_df["trigger_impact_score"], cfg["label_quantile"]))

    labels_df = pd.DataFrame(
        {
            "bank_id": state["node_ids"],
            "trigger_impact_score": trigger_df["trigger_impact_score"].to_numpy(),
        }
    )

    n_positive = max(1, int(math.ceil((1.0 - cfg["label_quantile"]) * n_nodes)))
    ranked = labels_df.sort_values(["trigger_impact_score", "bank_id"], ascending=[False, True]).reset_index()
    positive_idx = ranked.loc[: n_positive - 1, "index"].to_numpy()

    labels_df["systemically_important"] = 0
    labels_df.loc[positive_idx, "systemically_important"] = 1

    label_threshold = float(ranked.loc[n_positive - 1, "trigger_impact_score"])

    summary_df = pd.DataFrame(
        [
            {
                "quarter": quarter,
                "n_nodes": n_nodes,
                "n_edges": state["n_edges"],
                "n_scenarios": int(len(scenario_df)),
                "n_backfill": int(len(backfill_records)),
                "impact_score_mean": float(scenario_df["impact_score"].mean()),
                "impact_score_q90": float(scenario_df["impact_score"].quantile(0.90)),
                "default_count_mean": float(scenario_df["default_count"].mean()),
                "default_share_mean": float(scenario_df["default_share"].mean()),
                "loss_exposure_mean": float(scenario_df["loss_exposure"].mean()),
                "loss_liquidity_mean": float(scenario_df["loss_liquidity"].mean()),
                "max_default_count": int(scenario_df["default_count"].max()),
                "label_threshold": label_threshold,
                "quantile_threshold_raw": raw_quantile_threshold,
                "positive_rate": float(labels_df["systemically_important"].mean()),
                "n_positive": int(labels_df["systemically_important"].sum()),
                "negative_weight_policy": negative_weight_policy,
                "n_negative_input_edges": n_negative_input_edges,
                "n_edges_after_policy": n_edges_after_policy,
            }
        ]
    )

    label_path = OUTPUT_ROOT / "systemic_labels" / f"{quarter}_labels.csv"
    scenario_path = OUTPUT_ROOT / "scenario_logs" / f"{quarter}_scenarios.parquet"
    summary_path = OUTPUT_ROOT / "summaries" / f"{quarter}_summary.csv"

    labels_df.to_csv(label_path, index=False)
    _write_scenario_log(scenario_df, scenario_path)
    summary_df.to_csv(summary_path, index=False)

    return labels_df, scenario_df, summary_df


In [ ]:
def run_data_integrity_checks(cfg: dict[str, Any]) -> pd.DataFrame:
    """Validate quarter files and return integrity table."""
    rows = []
    for quarter in cfg["quarters"]:
        nodes_df, edges_df, node_index_map = load_quarter_data(quarter, cfg)

        diagnostics = edges_df.attrs.get("negative_weight_diagnostics", {})
        unique_nodes_ok = not nodes_df["index"].duplicated().any()
        endpoint_ok = edges_df["Sourceid"].isin(node_index_map).all() and edges_df["Targetid"].isin(node_index_map).all()
        non_negative_ok = (edges_df["Weights"] >= 0).all()
        node_cols_ok = "index" in nodes_df.columns

        rows.append(
            {
                "quarter": quarter,
                "n_nodes": int(len(nodes_df)),
                "n_edges": int(len(edges_df)),
                "unique_nodes_ok": bool(unique_nodes_ok),
                "edge_endpoint_ok": bool(endpoint_ok),
                "non_negative_weights_ok": bool(non_negative_ok),
                "node_columns_ok": bool(node_cols_ok),
                "negative_weight_policy": diagnostics.get("negative_weight_policy", cfg.get("negative_weight_policy", "split_by_sign")),
                "n_negative_input_edges": int(diagnostics.get("n_negative_input_edges", 0)),
                "n_edges_after_policy": int(diagnostics.get("n_edges_after_policy", len(edges_df))),
            }
        )

    integrity_df = pd.DataFrame(rows)
    strict_checks = ["unique_nodes_ok", "edge_endpoint_ok", "non_negative_weights_ok", "node_columns_ok"]
    if not integrity_df[strict_checks].all().all():
        raise AssertionError("Data integrity checks failed for one or more quarters")
    return integrity_df


integrity_df = run_data_integrity_checks(CONFIG)
integrity_df.head()


In [ ]:
def run_negative_weight_policy_unit_checks() -> pd.DataFrame:
    """Sanity-check signed-weight preprocessing policies."""
    sample = pd.DataFrame(
        {
            "Sourceid": [1, 2, 4, 7],
            "Targetid": [2, 3, 5, 8],
            "Weights": [100.0, -40.0, -5.5, 0.0],
        }
    )

    split_df = apply_negative_weight_policy(sample, "split_by_sign")
    ignore_df = apply_negative_weight_policy(sample, "ignore_negative")

    assert (split_df["Weights"] >= 0).all(), "split_by_sign must remove negative weights"
    assert (ignore_df["Weights"] >= 0).all(), "ignore_negative must remove negative weights"

    reversed_edge = split_df[(split_df["Sourceid"] == 3) & (split_df["Targetid"] == 2) & np.isclose(split_df["Weights"], 40.0)]
    assert not reversed_edge.empty, "split_by_sign must reverse negative edges"

    assert len(ignore_df) == int((sample["Weights"] >= 0).sum()), "ignore_negative must keep only non-negative edges"

    return pd.DataFrame(
        [
            {
                "policy": "split_by_sign",
                "n_edges": int(len(split_df)),
                "min_weight": float(split_df["Weights"].min()),
            },
            {
                "policy": "ignore_negative",
                "n_edges": int(len(ignore_df)),
                "min_weight": float(ignore_df["Weights"].min()),
            },
        ]
    )


def run_engine_sanity_checks(cfg: dict[str, Any], quarter: str = "2019Q1") -> dict[str, Any]:
    """Run minimal correctness checks for cascade logic."""
    nodes_df, edges_df, node_index_map = load_quarter_data(quarter, cfg)
    state = build_network_state(nodes_df, edges_df, node_index_map, cfg)
    n = state["n_nodes"]
    rng = np.random.default_rng(quarter_seed(cfg["rng_seed"], quarter) + 999)

    base_spec = {
        "scenario_id": "sanity_no_shock",
        "shock_type": "single",
        "seed_banks": [],
        "shock_values": [],
        "lgd": cfg["lgd"],
        "max_rounds": cfg["max_rounds"] or n,
    }
    no_shock = run_scenario(state, base_spec, rng)
    assert no_shock["default_count"] == 0, "No-shock scenario should not create defaults"

    test_seed = [0]
    mild = run_scenario(
        state,
        {
            "scenario_id": "sanity_mild",
            "shock_type": "single",
            "seed_banks": test_seed,
            "shock_values": [0.20],
            "lgd": cfg["lgd"],
            "max_rounds": cfg["max_rounds"] or n,
        },
        rng,
    )
    severe = run_scenario(
        state,
        {
            "scenario_id": "sanity_severe",
            "shock_type": "single",
            "seed_banks": test_seed,
            "shock_values": [0.60],
            "lgd": cfg["lgd"],
            "max_rounds": cfg["max_rounds"] or n,
        },
        rng,
    )

    assert severe["default_count"] >= mild["default_count"], "Higher shock should not reduce defaults"
    assert all(np.diff([0] + severe["default_count_history"]) >= 0), "Default count must be monotonic"
    assert severe["rounds"] <= n, "Scenario must stop at or before max_rounds"

    return {
        "quarter": quarter,
        "negative_weight_policy": cfg.get("negative_weight_policy", "split_by_sign"),
        "no_shock_default_count": no_shock["default_count"],
        "mild_default_count": mild["default_count"],
        "severe_default_count": severe["default_count"],
        "mild_rounds": mild["rounds"],
        "severe_rounds": severe["rounds"],
    }


policy_unit_checks = run_negative_weight_policy_unit_checks()
policy_unit_checks

sanity_results = run_engine_sanity_checks(CONFIG, quarter="2019Q1")
sanity_results


In [ ]:
def validate_label_outputs(labels_df: pd.DataFrame, expected_n_nodes: int, quantile_target: float) -> None:
    """Validate label output format and distribution."""
    assert len(labels_df) == expected_n_nodes, "One label per node is required"
    assert labels_df["systemically_important"].isin([0, 1]).all(), "Labels must be binary"
    assert np.isfinite(labels_df["trigger_impact_score"]).all(), "Trigger scores must be finite"

    expected_positive = max(1, int(math.ceil((1.0 - quantile_target) * expected_n_nodes)))
    observed_positive = int(labels_df["systemically_important"].sum())
    assert observed_positive == expected_positive, (
        f"Unexpected positive count: observed={observed_positive}, expected={expected_positive}"
    )


In [ ]:
# Smoke run (2 quarters, 200 scenarios each)
RUN_SMOKE = False

if RUN_SMOKE:
    smoke_cfg = copy.deepcopy(CONFIG)
    smoke_cfg["quarters"] = ["2019Q1", "2020Q1"]
    smoke_cfg["n_scenarios"] = 200

    smoke_summaries = []
    for quarter in smoke_cfg["quarters"]:
        labels_df, scenario_df, summary_df = run_quarter_simulations(quarter, smoke_cfg)

        expected_nodes = len(load_quarter_data(quarter)[0])
        validate_label_outputs(labels_df, expected_nodes, smoke_cfg["label_quantile"])
        smoke_summaries.append(summary_df)
        print(f"Smoke finished: {quarter} -> scenarios={len(scenario_df)} positives={labels_df['systemically_important'].sum()}")

    smoke_summary_df = pd.concat(smoke_summaries, ignore_index=True)
    display(smoke_summary_df)
else:
    print("RUN_SMOKE is False. Set to True to execute smoke run.")


In [ ]:
# Full run (all quarters, 10,000 scenarios each)
RUN_FULL = False

if RUN_FULL:
    full_summaries = []
    for quarter in CONFIG["quarters"]:
        labels_df, scenario_df, summary_df = run_quarter_simulations(quarter, CONFIG)

        expected_nodes = len(load_quarter_data(quarter)[0])
        validate_label_outputs(labels_df, expected_nodes, CONFIG["label_quantile"])

        full_summaries.append(summary_df)
        print(
            f"Completed {quarter}: scenarios={len(scenario_df)} ",
            f"positive_rate={summary_df.loc[0, 'positive_rate']:.4f}",
        )

    cross_quarter_df = pd.concat(full_summaries, ignore_index=True)
    cross_path = OUTPUT_ROOT / "summaries" / "all_quarters_summary.csv"
    cross_quarter_df.to_csv(cross_path, index=False)
    print(f"Saved cross-quarter summary: {cross_path}")
else:
    print("RUN_FULL is False. Set to True to execute all 29 quarters.")


In [ ]:
# Cross-quarter analysis and plots
summary_files = sorted((OUTPUT_ROOT / "summaries").glob("*_summary.csv"))
if summary_files:
    summary_df = pd.concat([pd.read_csv(path) for path in summary_files], ignore_index=True)
    summary_df = summary_df.sort_values("quarter").reset_index(drop=True)
    display(summary_df.head())

    if HAS_PLOTS:
        plt.figure(figsize=(12, 4))
        sns.barplot(data=summary_df, x="quarter", y="positive_rate", color="#4C78A8")
        plt.xticks(rotation=90)
        plt.title("Systemically Important Positive Rate by Quarter")
        plt.tight_layout()
        plt.savefig(OUTPUT_ROOT / "plots" / "positive_rate_by_quarter.png", dpi=150)
        plt.show()

        plt.figure(figsize=(12, 4))
        sns.lineplot(data=summary_df, x="quarter", y="impact_score_mean", marker="o")
        plt.xticks(rotation=90)
        plt.title("Mean Impact Score by Quarter")
        plt.tight_layout()
        plt.savefig(OUTPUT_ROOT / "plots" / "impact_mean_by_quarter.png", dpi=150)
        plt.show()
    else:
        print("matplotlib/seaborn not available, skipping plots.")

    selected_quarter = "2022Q4"
    labels_path = OUTPUT_ROOT / "systemic_labels" / f"{selected_quarter}_labels.csv"
    if labels_path.exists():
        top20 = pd.read_csv(labels_path).sort_values("trigger_impact_score", ascending=False).head(20)
        display(top20)
    else:
        print(f"Labels not found for {selected_quarter}: {labels_path}")
else:
    print("No summary files found yet. Run smoke or full execution first.")
